# SIMULATION OF ELECTROMAGNETIC WAVE IN RING RESONATOR WAVEGUIDE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image

# KONSTANTA FISIK
c0 = 3e8
mu0 = 4 * np.pi * 1e-7
eps0 = 1 / (mu0 * c0**2)

# Spektrum target
lambda_min = 1.3e-6   # 1.3 μm
lambda_max = 1.7e-6   # 1.7 μm
f_min = c0 / lambda_max
f_max = c0 / lambda_min
f_center = 0.5 * (f_min + f_max)

# PARAMETER GRID
dx = dy = 0.1e-6   # 0.1 µm grid spacing
Nx, Ny = 300, 150
Nt = 10
duration = 30
dt = 1 / (c0 * np.sqrt(1/dx**2 + 1/dy**2)) * 0.99

# INISIALISASI MEDAN
Ez = np.zeros((Nx, Ny))
Ez_prev = np.zeros((Nx, Ny))
Hx = np.zeros((Nx, Ny))
Hy = np.zeros((Nx, Ny))

chx = dt / (mu0 * dy)
chy = dt / (mu0 * dx)

# MATERIAL MAP
R = 40
thickness = 6
gap = 7
gap_mikro = gap * dx #dalam m
gap_m = gap_mikro * 1e6 #dalam μm
n_core = 3.45
n_clad = 1.44

eps = eps0 * n_clad**2 * np.ones((Nx, Ny))
x0, y0 = Nx//2, Ny//2

# RING RESONATOR
for i in range(Nx):
    for j in range(Ny):
        r = np.sqrt((i-x0)**2 + (j-y0)**2)
        if R - thickness/2 < r < R + thickness/2:
            eps[i,j] = eps0 * n_core**2

# BUS WAVEGUIDE
bus_y = y0 + (R + gap)
for i in range(Nx):
    for t in range(-thickness//2, thickness//2 + 1):
        yy = bus_y + t
        if 0 <= yy < Ny:
            eps[i, yy] = eps0 * n_core**2

# Sumber
source_x, source_y = 1, bus_y
pulse_width = 10
pulse_center = 20

# MUR ABC
def mur_boundary(Ez, Ez_prev, c, dt, dx):
    coef = (c * dt - dx) / (c * dt + dx)
    Ez[0, 1:-1]   = Ez_prev[1, 1:-1]   + coef * (Ez[1, 1:-1]   - Ez_prev[0, 1:-1])
    Ez[-1, 1:-1]  = Ez_prev[-2, 1:-1]  + coef * (Ez[-2, 1:-1]  - Ez_prev[-1, 1:-1])
    Ez[1:-1, 0]   = Ez_prev[1:-1, 1]   + coef * (Ez[1:-1, 1]   - Ez_prev[1:-1, 0])
    Ez[1:-1, -1]  = Ez_prev[1:-1, -2]  + coef * (Ez[1:-1, -2]  - Ez_prev[1:-1, -1])
    return Ez

# RUN FDTD
n_steps = 2000
frames = []
source_signal = []
sample_every = 5

# TITIK MONITOR UNTUK ANALISIS SPEKTRUM ---
ring_probe = (x0, y0 + R)
bus_probe  = (Nx-5, bus_y)

bus_monitor = []
ring_monitor = []

for n in range(n_steps):
    # update H fields
    Hx[:, :-1] -= chx * (Ez[:, 1:] - Ez[:, :-1])
    Hy[:-1, :] += chy * (Ez[1:, :] - Ez[:-1, :])

    curl_H = np.zeros_like(Ez)
    curl_H[1:, 1:] = ((Hy[1:, 1:] - Hy[:-1, 1:]) / dx -
                      (Hx[1:, 1:] - Hx[1:, :-1]) / dy)

    Ez_prev[:] = Ez
    Ez += (dt / eps) * curl_H

    # sumber Gaussian broadband
    s = np.exp(-((n - pulse_center) / pulse_width)**2)
    Ez[source_x, source_y] += s
    source_signal.append(s)

    # rekam sinyal monitor
    bus_monitor.append(Ez[bus_probe])
    ring_monitor.append(Ez[ring_probe])

    # boundary Mur
    Ez = mur_boundary(Ez, Ez_prev, c0, dt, dx)

    if n % sample_every == 0:
        frames.append(Ez.copy())


# --- ANIMASI ---
fig, ax = plt.subplots()
extent = [0, Nx * dx * 1e6, 0, Ny * dy * 1e6]
im_ez = ax.imshow(np.flipud(frames[0].T), cmap='seismic',
                  vmin=-0.02, vmax=0.02, extent=extent)
cbar = plt.colorbar(im_ez, ax=ax)
cbar.set_label("Ez (arb. units)")

eps_binary = (eps > eps0 * n_clad**2).astype(float)
ax.contour(np.flipud(eps_binary.T), levels=[0.5], colors="black",
           linewidths=0.7, extent=extent, origin="upper")

ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_title(f"Simulasi FDTD Ring Resonator gap = {gap_m} μm")

def update(i):
    im_ez.set_data(np.flipud(frames[i].T))
    return [im_ez]

ani = FuncAnimation(fig, update, frames=len(frames),
                    interval=duration * 5000 / Nt, blit=False)

gif_path = "simulasi.gif"
ani.save(gif_path, writer=PillowWriter(fps=20))
print("GIF tersimpan:", gif_path)
display(Image(filename=gif_path))


# ============================
#   ANALISIS SPEKTRUM
# ============================

bus_monitor = np.array(bus_monitor)
ring_monitor = np.array(ring_monitor)
source_signal = np.array(source_signal)

# FFT
N = len(source_signal)
freqs = np.fft.rfftfreq(N, dt)
omega = 2 * np.pi * freqs

SRC = np.abs(np.fft.rfft(source_signal))
OUT_bus = np.abs(np.fft.rfft(bus_monitor))
OUT_ring = np.abs(np.fft.rfft(ring_monitor))

# Konversi frekuensi → panjang gelombang λ = c/f
wavelength = c0 / freqs
mask = (wavelength >= 1e-6) & (wavelength <= 2e-6)

λ = wavelength[mask] * 1e6           # dalam μm
T_bus = (OUT_bus[mask] / SRC[mask])**2
T_ring = (OUT_ring[mask] / SRC[mask])**2

# ============================
#   PLOTTING SPEKTRUM
# ============================

plt.figure(figsize=(7,5))
plt.plot(λ, T_bus, label="Transmisi Bus")
plt.plot(λ, T_ring, label="Transmisi Ring")
plt.gca().invert_xaxis()
plt.xlabel("Wavelength (μm)")
plt.ylabel("Transmission (arb.)")
plt.title(f"Transmisi Ring Resonator gap = {gap_m} μm")
plt.legend()
plt.grid(True)
plt.show()